# Train IQ and spectrogram modrec models on the same TorchSig data

This GPU-oriented experiment makes the comparison from the companion notebook concrete. It generates each TorchSig split **once as IQ**, then presents the exact same examples to two classifiers:

- the repository's 1-D EfficientNet-B0 receiving I and Q channels;
- the repository's 2-D EfficientNet-B0 receiving a log-power spectrogram.

The selected labels mix phase/amplitude-sensitive linear modulations with time-frequency-oriented FSK and chirp signals. The goal is not to declare a universal winner, but to inspect which classes and SNR ranges favor each representation. A CUDA GPU is assumed for training.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, default_collate

from torchsig_models.models.iq_models.efficientnet import efficientnet_b0 as iq_efficientnet_b0
from torchsig_models.models.spectrogram_models.efficientnet import efficientnet_b0 as spectrogram_efficientnet_b0
from torchsig_models.utils.training import compute_num_params, set_deterministic, train_validate

from torchsig.datasets.datasets import StaticTorchSigDataset, TorchSigIterableDataset
from torchsig.transforms.transforms import ComplexTo2D
from torchsig.utils.data_loading import WorkerSeedingDataLoader
from torchsig.utils.defaults import TorchSigDefaults
from torchsig.utils.writer import DatasetCreator

SEED = 2026
set_deterministic(SEED)
torch.set_float32_matmul_precision("high")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for a CUDA GPU.")
device = torch.device("cuda")
print(torch.cuda.get_device_name(), torch.__version__)

## Experiment configuration

Increase the split sizes or epochs for a publication-quality comparison. The defaults are intended to demonstrate a meaningful experiment on a single GPU. Static data are kept under `runs/` and reused unless `REGENERATE` is set.

In [ ]:
SIGNALS = ["bpsk", "qpsk", "16qam", "2fsk", "4fsk", "lfm-radar"]
NUM_CLASSES = len(SIGNALS)
SPLIT_SIZES = {"train": 12_000, "val": 2_000, "test": 4_000}
SPLIT_SEEDS = {"train": 3101, "val": 3102, "test": 3103}
DATA_ROOT = Path("runs/iq_vs_spectrogram_modrec/data")
REGENERATE = False
NUM_IQ_SAMPLES = 4096
FFT_SIZE = 128
FFT_HOP = 32
BATCH_SIZE = 128
NUM_WORKERS = 8
EPOCHS = 15
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-4
SNR_BINS = [-10, 0, 5, 10, 15, 20, 30]

metadata = TorchSigDefaults().default_dataset_metadata.copy()
metadata.update({
    "sample_rate": 10_000_000,
    "num_iq_samples_dataset": NUM_IQ_SAMPLES,
    "num_signals_min": 1,
    "num_signals_max": 1,
    "cochannel_overlap_probability": 0.0,
    "snr_db_min": -5.0,
    "snr_db_max": 25.0,
    "noise_power_db": 0.0,
    "signal_duration_in_samples_min": 3600,
    "signal_duration_in_samples_max": NUM_IQ_SAMPLES,
    "bandwidth_min": 750_000,
    "bandwidth_max": 2_500_000,
    "signal_center_freq_min": -2_500_000,
    "signal_center_freq_max": 2_500_000,
    "frequency_min": -5_000_000,
    "frequency_max": 5_000_000,
})

## Generate one shared IQ dataset

The saved target includes SNR so performance can be stratified later. Most importantly, no independently sampled spectrogram dataset is created: both models see the same signal identities, channel conditions, offsets, and noise realizations.

In [ ]:
def create_split(name: str, length: int, seed: int) -> Path:
    root = DATA_ROOT / name
    if root.exists() and not REGENERATE:
        print(f"Reusing {root}")
        return root

    split_metadata = metadata.copy()
    source = TorchSigIterableDataset(
        metadata=split_metadata, seed=seed,
        transforms=[ComplexTo2D()],
        signal_generators=SIGNALS,
        target_labels=["class_index", "snr_db"],
    )
    generation_loader = WorkerSeedingDataLoader(
        source, batch_size=64, num_workers=NUM_WORKERS, collate_fn=list
    )
    DatasetCreator(
        dataloader=generation_loader, root=str(root), overwrite=REGENERATE, dataset_length=length
    ).create()
    return root

split_roots = {
    name: create_split(name, length, SPLIT_SEEDS[name])
    for name, length in SPLIT_SIZES.items()
}
raw_splits = {
    name: StaticTorchSigDataset(root=str(root), target_labels=["class_index", "snr_db"])
    for name, root in split_roots.items()
}
print({name: len(dataset) for name, dataset in raw_splits.items()})

## Paired representation datasets

The wrappers normalize each example and preserve `(label, SNR)` for evaluation. Random phase rotation is used only for IQ training, teaching the 1-D model that absolute phase is a nuisance. The spectrogram already has that invariance because it uses magnitude.

In [ ]:
def unpack_target(target):
    label, snr = target
    return int(np.asarray(label).reshape(-1)[0]), float(np.asarray(snr).reshape(-1)[0])

class RepresentationDataset(Dataset):
    def __init__(self, base, representation: str, augment: bool = False):
        if representation not in {"iq", "spectrogram"}:
            raise ValueError(representation)
        self.base = base
        self.representation = representation
        self.augment = augment
        self.stft_window = torch.hann_window(FFT_SIZE)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        iq, target = self.base[index]
        iq = torch.as_tensor(iq, dtype=torch.float32)
        label, snr = unpack_target(target)
        if not 0 <= label < NUM_CLASSES:
            raise ValueError(f"Unexpected class index {label}; regenerate with SIGNALS={SIGNALS}")

        if self.augment:
            phase = torch.rand(()) * (2 * torch.pi)
            i, q = iq[0].clone(), iq[1].clone()
            iq = torch.stack((i * phase.cos() - q * phase.sin(), i * phase.sin() + q * phase.cos()))

        iq = iq / iq.square().mean().sqrt().clamp_min(1e-6)
        if self.representation == "iq":
            x = iq
        else:
            z = torch.complex(iq[0], iq[1])
            stft = torch.stft(
                z, n_fft=FFT_SIZE, hop_length=FFT_HOP, win_length=FFT_SIZE,
                window=self.stft_window, center=False, return_complex=True,
            )
            x = torch.log1p(stft.abs().square()).unsqueeze(0)
            x = (x - x.mean()) / x.std().clamp_min(1e-6)
        return x, label, snr

datasets = {
    rep: {
        split: RepresentationDataset(raw, rep, augment=(rep == "iq" and split == "train"))
        for split, raw in raw_splits.items()
    }
    for rep in ("iq", "spectrogram")
}
def classification_collate(batch):
    """Drop SNR metadata from batches consumed by train_validate."""
    inputs, labels, _ = default_collate(batch)
    return inputs, labels

loaders = {
    rep: {
        split: DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=(split == "train"),
            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
            collate_fn=classification_collate if split != "test" else None,
        )
        for split, ds in splits.items()
    }
    for rep, splits in datasets.items()
}
print("IQ shape:", datasets["iq"]["train"][0][0].shape)
print("Spectrogram shape:", datasets["spectrogram"]["train"][0][0].shape)

## Repository EfficientNet-B0 models

Both branches use the EfficientNet-B0 implementations included in this repository: the 1-D adaptation for IQ and the 2-D implementation for spectrograms. They use the same dropout settings, optimizer, epoch budget, and selection rule. Parameter counts are printed, but the models do not have identical compute because 1-D and 2-D operations act on different grids.

In [ ]:
models = {
    "iq": iq_efficientnet_b0(
        num_classes=NUM_CLASSES, drop_path_rate=0.1, drop_rate=0.1
    ),
    "spectrogram": spectrogram_efficientnet_b0(
        num_classes=NUM_CLASSES, input_channels=1,
        drop_path_rate=0.1, drop_rate=0.1, normalize=False,
    ),
}
for name, model in models.items():
    count = compute_num_params(model)
    print(f"{name:12}: {count:,} parameters")

## Train both models

Automatic mixed precision is enabled. The best validation checkpoint is restored before testing. Training time includes dataloading and, for the spectrogram branch, STFT preprocessing.

In [ ]:
def train_model(model, representation):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    run_dir = DATA_ROOT.parent / representation
    started = time.perf_counter()

    pl_model, metrics = train_validate(
        train_loader=loaders[representation]["train"],
        val_loader=loaders[representation]["val"],
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        max_epochs=EPOCHS,
        num_classes=NUM_CLASSES,
        metrics_dir=run_dir / "metrics",
        checkpoint_dir=run_dir / "checkpoints",
        accelerator="gpu",
        devices=1,
        precision="16-mixed",
        use_distributed_sampler=False,
        logger=False,
    )

    checkpoint_path = max(
        (run_dir / "checkpoints").glob("best-epoch*.ckpt"),
        key=lambda path: path.stat().st_mtime,
    )
    checkpoint = torch.load(
        checkpoint_path, map_location=device, weights_only=False
    )
    pl_model.load_state_dict(checkpoint["state_dict"])

    history = {
        "train_loss": metrics.train_metrics.history["loss"],
        "val_accuracy": metrics.val_metrics.history["accuracy"],
        "seconds": time.perf_counter() - started,
    }
    return pl_model.model.to(device), history

histories = {}
for representation in ("iq", "spectrogram"):
    print(f"\\nTraining {representation}")
    models[representation], histories[representation] = train_model(
        models[representation], representation
    )


## Paired test evaluation

Overall accuracy alone can hide the expected effect. The confusion matrices reveal which modulation families benefit from each representation, while the SNR curves show where those differences appear.

In [ ]:
@torch.inference_mode()
def collect_predictions(model, loader):
    model.eval()
    predictions, targets, snrs = [], [], []
    for x, y, snr in loader:
        predictions.append(model(x.to(device, non_blocking=True)).argmax(1).cpu())
        targets.append(y)
        snrs.append(torch.as_tensor(snr))
    return torch.cat(predictions).numpy(), torch.cat(targets).numpy(), torch.cat(snrs).numpy()

def confusion_matrix(targets, predictions):
    matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    np.add.at(matrix, (targets, predictions), 1)
    return matrix

results = {
    rep: collect_predictions(models[rep], loaders[rep]["test"])
    for rep in ("iq", "spectrogram")
}
assert np.array_equal(results["iq"][1], results["spectrogram"][1])
assert np.allclose(results["iq"][2], results["spectrogram"][2])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, rep in zip(axes, ("iq", "spectrogram")):
    pred, target, _ = results[rep]
    matrix = confusion_matrix(target, pred)
    normalized = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
    image = ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
    ax.set(title=f"{rep}: accuracy={(pred == target).mean():.2%}", xlabel="Predicted", ylabel="True")
    ax.set_xticks(range(NUM_CLASSES), SIGNALS, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), SIGNALS)
    for row in range(NUM_CLASSES):
        for col in range(NUM_CLASSES):
            ax.text(col, row, f"{normalized[row, col]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=axes, shrink=.8, label="Row-normalized fraction")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for rep in ("iq", "spectrogram"):
    axes[0].plot(histories[rep]["val_accuracy"], marker="o", label=rep)
    prediction, target, snr = results[rep]
    bin_accuracy = []
    bin_centers = []
    for lower, upper in zip(SNR_BINS[:-1], SNR_BINS[1:]):
        selected = (snr >= lower) & (snr < upper)
        if selected.any():
            bin_accuracy.append((prediction[selected] == target[selected]).mean())
            bin_centers.append((lower + upper) / 2)
    axes[1].plot(bin_centers, bin_accuracy, marker="o", label=rep)
axes[0].set(title="Validation learning curves", xlabel="Epoch", ylabel="Accuracy")
axes[1].set(title="Paired test accuracy by SNR", xlabel="SNR (dB)", ylabel="Accuracy", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=.25); ax.legend()
plt.tight_layout()

for rep in ("iq", "spectrogram"):
    prediction, target, _ = results[rep]
    print(f"{rep:12}: test={(prediction == target).mean():.2%}, training={histories[rep]['seconds']/60:.1f} min")

## Interpreting the result

Look for three patterns rather than only the headline score:

1. **BPSK/QPSK/16-QAM confusions:** IQ retains phase and amplitude geometry, so it often has an advantage among these classes—provided augmentation and architecture handle carrier phase/frequency nuisance.
2. **2-FSK/4-FSK/LFM performance:** frequency trajectories are explicit in a spectrogram, so the 2-D model may learn these efficiently.
3. **SNR dependence:** a representation advantage may reverse or disappear as noise increases.

This remains a baseline, not a controlled scientific conclusion. For a stronger study, repeat several seeds; tune each architecture fairly; match measured inference cost; sweep FFT size/hop; add realistic receiver impairments; report confidence intervals and per-class accuracy; and test complex STFT or late fusion. If one representation wins only because it received more tuning or a better-sized model, the comparison is not informative.